##GroupDNA — WhatsApp Group Chat Analyzer

Minor Project — Week-02 Industry-Graded Minor Project<br>
Name: V.Swapnith Chowdary<br>
Project: SpendDNA<br>
Dataset: Rahul_transaction.csv<br>
Date: 26 September 2026



##1. The Vision - What you're Actually Building

In [ ]:
# SpendDNA
# Analyse Rahul's 6 months of UPI transactions
# and generate a financial spending report.

##2. The Problem Statement

In [ ]:
import pandas as pd
import numpy as np

##3. Understanding the input File

In [ ]:
# Read the CSV file

df = pd.read_csv("rahul_transactions.csv")

print(df.head())
print(df.columns)
print(df.shape)

          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode        Ref  
0  678275.0   UPI  TXN190872  
1  681007.0   UPI  TXN143064  
2  484728.0  NEFT  TXN246316  
3 -748745.0   UPI  TXN569226  
4  680737.0   UPI  TXN968962  
Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref'],
      dtype='object')
(1328, 8)


##3.1 Four data formats

In [ ]:
df["date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True
)

##3.2 Three Amount formats

In [ ]:
df["amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)

##3.3 Type field variants

In [ ]:
df["type"] = (
    df["Type"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["type"] = df["type"].replace({
    "dr": "debit",
    "cr": "credit"
})

##3.4 Multi-variant vendor descriptions

In [ ]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Duplicates removed:", before - after)
print("Remaining transactions:", after)

Duplicates removed: 18
Remaining transactions: 1310


##4. The Provided Dataset - rahul_transactions.csv

In [ ]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 transactions:")
print(df.head())

print("\nData types:")
print(df.dtypes)
df["hour"] = df["Time"].astype(str).str[:2].astype(int)

df["month"] = df["date"].dt.month

df["month_name"] = df["date"].dt.strftime("%b")

df["day_of_week"] = df["date"].dt.day_name()

Dataset shape: (1310, 15)

Columns:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref', 'date', 'amount', 'type', 'hour', 'month', 'month_name', 'day_of_week']

First 5 transactions:
          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode        Ref       date   amount    type  hour  month  \
0  678275.0   UPI  TXN190872 2024-01-01   2462.0   debit     3    1.0   
1  681007.0   UPI  TXN143064        NaT     50.0   debit     5    NaN   
2  484728.0  NEFT  TXN246316        NaT  84728.0  credit     9    NaN   
3 -748745.0   UPI  TXN569226 2024-01-01   1828.0   debit

##5. New Concepts You Need
##5.1 The .dt accessor

In [ ]:
df["date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True
)

df["month"] = df["date"].dt.month

df["day_of_week"] = df["date"].dt.day_name()

df["week_number"] = df["date"].dt.isocalendar().week

df["hour"] = df["Time"].str[:2].astype(int)

##5.2 Z-Score anomaly detection

In [ ]:
category_mean = (
    df.groupby("Description")["amount"]
    .transform("mean")
)

category_std = (
    df.groupby("Description")["amount"]
    .transform("std")
)

df["z_score"] = (
    (df["amount"] - category_mean)
    / category_std
)

anomalies = df[df["z_score"] > 2]

print(anomalies)

             Date   Time               Description   Type      Amount  \
169   24 Jan 2024  18:43            FLIPKART INDIA     DR       16260   
223    2024-02-01  09:20       INSTAMART BANGALORE  Debit      791.00   
269      07/02/24  17:28    UPI-AMAZONPAY@HDFCBANK  Debit      ₹21986   
414    2024-02-26  20:09  POS BANGALORE RESTAURANT     DR    8,383.00   
450     02-Mar-24  11:58                 AMAZON IN  Debit       18273   
464      04/03/24  16:12              POS TRUFFLES     DR       ₹7441   
653      31/03/24  20:01         POS MEGHANA FOODS     DR       ₹7931   
726   10 Apr 2024  13:39     POS HP PETROL STATION     DR    8,350.00   
736     11-Apr-24  16:08           KIRANAKART TECH     DR      740.00   
875     30-Apr-24  14:19          Amazon Pay India     DR   14,886.00   
1144    05-Jun-24  20:32          Amazon Pay India  Debit   14,874.00   
1271     22/06/24  12:46               POS DINEOUT  Debit       ₹7935   
1291  25 Jun 2024  14:21          Amazon Pay India 

##6. Core Features - What you must Build (Mandatory)

##Feature 1 - The Transaction Parser

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("rahul_transactions.csv")

# Date
df["date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True
)

# Amount
df["amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)

# Type
df["type"] = (
    df["Type"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["type"] = df["type"].replace({
    "dr": "debit",
    "cr": "credit"
})

# Remove duplicates
df = df.drop_duplicates()

# Remove invalid dates/amounts
df = df.dropna(
    subset=["date", "amount"]
)

print("Parsed transactions:", len(df))
print("Duplicates removed.")
print("Parser completed.")

Parsed transactions: 143
Duplicates removed.
Parser completed.


##Feature 2 - Vendor Extractor

In [ ]:
vendor_dictionary = {
    "Swiggy": ["SWIGGY", "BUNDL"],
    "Zomato": ["ZOMATO"],
    "Amazon": ["AMAZON", "AMZN"],
    "Zepto": ["ZEPTO"],
    "Zerodha": ["ZERODHA"],
    "Myntra": ["MYNTRA"],
    "BookMyShow": ["BOOKMYSHOW"],
    "Uber": ["UBER"],
    "Ola": ["OLA"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "YouTube": ["YOUTUBE"],
    "Flipkart": ["FLIPKART"],
    "Blinkit": ["BLINKIT"],
    "BigBasket": ["BIGBASKET"],
    "DMart": ["DMART"],
    "Reliance": ["RELIANCE"],
    "Shell": ["SHELL"],
    "HP": ["HPCL"],
    "PVR": ["PVR"],
    "BookMyShow": ["BOOKMYSHOW"]
}


def extract_vendor(description):

    text = str(description).upper()

    # ATM
    if "ATM-WDL" in text:
        return "Cash Withdrawal"

    # P2P
    if "UPI-" in text and "@" in text:
        known_vendor_found = False

        for vendor, keywords in vendor_dictionary.items():
            for keyword in keywords:
                if keyword in text:
                    known_vendor_found = True
                    return vendor

        return "P2P Transfer"

    # Normal vendors
    for vendor, keywords in vendor_dictionary.items():

        for keyword in keywords:

            if keyword in text:
                return vendor

    return "Uncategorised"


df["vendor_clean"] = df["Description"].apply(
    extract_vendor
)

print(df["vendor_clean"].value_counts().head(10))

vendor_clean
Uncategorised    36
Swiggy           28
P2P Transfer     17
Zomato           16
Ola               8
Uber              7
Zepto             5
Flipkart          4
Zerodha           4
Amazon            3
Name: count, dtype: int64


##Feature 3 - Catefory Tagger

In [ ]:
category_dictionary = {

    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",

    "Amazon": "E-commerce",
    "Myntra": "E-commerce",
    "Flipkart": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",

    "Starbucks": "Cafe",
    "CCD": "Cafe",

    "PVR": "Entertainment",
    "BookMyShow": "Entertainment",

    "Zerodha": "Investments",

    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "YouTube": "Subscriptions",

    "BigBasket": "Groceries",
    "DMart": "Groceries",

    "Shell": "Fuel",
    "HP": "Fuel",

    "Cash Withdrawal": "Cash Withdrawal",
    "P2P Transfer": "Personal Transfer"
}

df["category"] = (
    df["vendor_clean"]
    .map(category_dictionary)
    .fillna("Uncategorised")
)

print(df["category"].value_counts())

category
Food Delivery        44
Uncategorised        36
Personal Transfer    17
Transport            15
E-commerce            9
Quick Commerce        7
Groceries             4
Investments           4
Entertainment         3
Cash Withdrawal       3
Subscriptions         1
Name: count, dtype: int64


##Feature 4 - Spending Overview

In [ ]:
credits = df.loc[
    df["type"] == "credit",
    "amount"
].sum()

debits = df.loc[
    df["type"] == "debit",
    "amount"
].sum()

net_savings = credits - debits
savings_rate = (
    net_savings / credits * 100
    if credits != 0 else 0
)

print("\nSPENDING OVERVIEW:")
print("=" * 60)
top_categories = (
    df[df["type"] == "debit"]
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
print(top_categories)
print("\nTOP VENDORS:")

top_vendors = (
    df[df["type"] == "debit"]
    .groupby("vendor_clean")["amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
print(top_vendors)
print("\nTOTAL TRANSACTIONS:", len(df))


SPENDING OVERVIEW
category
Uncategorised        60430.0
Investments          60000.0
E-commerce           31220.0
Food Delivery        19617.0
Personal Transfer    15728.0
Name: amount, dtype: float64

TOP VENDORS
vendor_clean
Uncategorised    60430.0
Zerodha          60000.0
P2P Transfer     15728.0
Myntra           13629.0
Swiggy           12638.0
Name: amount, dtype: float64

TOTAL TRANSACTIONS: 143


##Feature 5 - Monthly Trend Analysis

In [ ]:
df["month"] = df["date"].dt.month
debit_df = df[df["type"] == "debit"].copy()

month_pivot = debit_df.pivot_table(
    values="amount",
    index="category",
    columns="month",
    aggfunc="sum",
    fill_value=0
)

print("\nMONTHLY TREND")
print("=" * 60)

print(month_pivot)

growth = {}

for category in month_pivot.index:

    first_month = month_pivot.loc[category, 1]
    last_month = month_pivot.loc[category, 6]

    if first_month != 0:
        percentage_change = (
            (last_month - first_month)
            / first_month
        ) * 100

        growth[category] = percentage_change

growth_series = pd.Series(growth).sort_values(
    ascending=False
)

print("\nBIGGEST GROWTH")
print(growth_series.head(1))

print("\nBIGGEST DECLINE")
print(growth_series.tail(1))


MONTHLY TREND
month                  1       2        3       4        5       6        7   \
category                                                                       
Cash Withdrawal       0.0     0.0      0.0     0.0      0.0     0.0      0.0   
E-commerce         5346.0   996.0   2147.0  3311.0      0.0  3000.0   1583.0   
Entertainment         0.0     0.0      0.0   343.0      0.0     0.0    562.0   
Food Delivery      1180.0  3454.0   2862.0   632.0   1159.0  1636.0   1686.0   
Groceries             0.0     0.0   1213.0     0.0      0.0     0.0   1257.0   
Investments           0.0     0.0      0.0     0.0      0.0     0.0  15000.0   
Personal Transfer  2123.0   355.0    501.0   447.0    166.0  1860.0   2315.0   
Quick Commerce      825.0   697.0      0.0     0.0    502.0     0.0    196.0   
Subscriptions         0.0     0.0      0.0     0.0      0.0     0.0      0.0   
Transport           826.0   540.0    439.0     0.0    431.0   134.0    351.0   
Uncategorised      4672.0

##Feature 6 - Time-of-Day Patterns

In [ ]:
df["hour"] = (
    df["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)

time_matrix = df.pivot_table(
    values="amount",
    index="category",
    columns="hour",
    aggfunc="sum",
    fill_value=0
)

print("\nTIME-OF-DAY PATTERNS")
print("=" * 60)

print(time_matrix)

food_delivery = df[
    df["category"] == "Food Delivery"
]

late_night_food = food_delivery[
    (food_delivery["hour"] >= 21) |
    (food_delivery["hour"] <= 2)
]

percentage_late_night = (
    len(late_night_food)
    / len(food_delivery)
    * 100
)

print(
    f"Food Delivery after 9 PM: "
    f"{percentage_late_night:.2f}%"
)


TIME-OF-DAY PATTERNS
hour                  0      1      2       3        4       5       6   \
category                                                                  
Cash Withdrawal      0.0    0.0    0.0     0.0      0.0     0.0     0.0   
E-commerce           0.0    0.0    0.0  2462.0      0.0     0.0     0.0   
Entertainment      562.0    0.0    0.0     0.0      0.0     0.0     0.0   
Food Delivery      394.0    0.0  318.0     0.0    413.0   537.0     0.0   
Groceries            0.0    0.0    0.0     0.0      0.0     0.0     0.0   
Investments          0.0    0.0    0.0     0.0  15000.0     0.0     0.0   
Personal Transfer    0.0  842.0    0.0     0.0      0.0  4496.0     0.0   
Quick Commerce       0.0    0.0  697.0     0.0      0.0     0.0     0.0   
Subscriptions        0.0    0.0    0.0     0.0      0.0     0.0     0.0   
Transport          431.0    0.0    0.0     0.0      0.0   425.0   322.0   
Uncategorised      433.0  620.0    0.0   766.0      0.0     0.0  1626.0   

ho

##Feature 7 - Anomaly Detection

In [ ]:
category_mean = (
    df.groupby("category")["amount"]
    .transform("mean")
)

category_std = (
    df.groupby("category")["amount"]
    .transform("std")
)

df["z_score"] = (
    (df["amount"] - category_mean)
    / category_std
)

anomalies = df[
    (df["type"] == "debit") &
    (df["z_score"] > 2)
].copy()

anomalies = anomalies.sort_values(
    "z_score",
    ascending=False
)

print("\nTOP ANOMALIES")
print("=" * 60)

print(
    anomalies[
        [
            "date",
            "vendor_clean",
            "category",
            "amount",
            "z_score"
        ]
    ].head(5)
)


TOP ANOMALIES
           date  vendor_clean           category   amount   z_score
1159 2024-08-06  P2P Transfer  Personal Transfer   4496.0  3.296749
70   2024-09-01        Myntra         E-commerce  10745.0  2.526528


##Feature 8 - Spending Archetype Detection

In [ ]:
total_debits = df.loc[
    df["type"] == "debit",
    "amount"
].sum()

credits = df.loc[
    df["type"] == "credit",
    "amount"
].sum()

net_savings = credits - total_debits
savings_rate = (
    net_savings / credits * 100
    if credits != 0 else 0
)

def percentage_for_category(category):

    amount = df.loc[
        (df["type"] == "debit") &
        (df["category"] == category),
        "amount"
    ].sum()

    if total_debits == 0:
        return 0

    return amount / total_debits * 100


food_percentage = (
    percentage_for_category("Food Delivery")
    + percentage_for_category("Restaurants")
    + percentage_for_category("Cafe")
)

quick_percentage = percentage_for_category(
    "Quick Commerce"
)

ecommerce_percentage = percentage_for_category(
    "E-commerce"
)

investment_percentage = percentage_for_category(
    "Investments"
)

transport_percentage = percentage_for_category(
    "Transport"
)

subscription_vendors = df[
    df["category"] == "Subscriptions"
]["vendor_clean"].nunique()

food_delivery_df = df[
    df["category"] == "Food Delivery"
]

late_food = food_delivery_df[
    (food_delivery_df["hour"] >= 21) |
    (food_delivery_df["hour"] <= 2)
]

if len(food_delivery_df) > 0:
    late_food_percentage = (
        len(late_food)
        / len(food_delivery_df)
        * 100
    )
else:
    late_food_percentage = 0


archetypes = []

if food_percentage > 25:
    archetypes.append(
        ("THE FOODIE", food_percentage)
    )

if quick_percentage > 15:
    archetypes.append(
        ("THE QUICK COMMERCE JUNKIE",
         quick_percentage)
    )

if ecommerce_percentage > 15:
    archetypes.append(
        ("THE SHOPAHOLIC",
         ecommerce_percentage)
    )

if investment_percentage > 15:
    archetypes.append(
        ("THE INVESTOR",
         investment_percentage)
    )

if late_food_percentage > 50:
    archetypes.append(
        ("THE LATE-NIGHT SNACKER",
         late_food_percentage)
    )

if transport_percentage > 10:
    archetypes.append(
        ("THE CAB COMMUTER",
         transport_percentage)
    )

if subscription_vendors >= 5:
    archetypes.append(
        ("THE SUBSCRIPTION LOVER",
         subscription_vendors)
    )

if savings_rate < 10:
    archetypes.append(
        ("THE YOLO SPENDER",
         savings_rate)
    )

if savings_rate > 40:
    archetypes.append(
        ("THE DISCIPLINED SAVER",
         savings_rate)
    )

print("\nRAHUL'S SPENDING ARCHETYPES")
print("=" * 60)

for name, metric in archetypes:
    print(f"-> {name} ({metric:.2f}%)")


RAHUL'S SPENDING ARCHETYPES
-> THE SHOPAHOLIC (15.21%)
-> THE INVESTOR (29.22%)
-> THE YOLO SPENDER (-20.70%)


##7. The Spending Archetype System

In [ ]:
# THE FOODIE
# Food Delivery + Restaurants + Cafe > 25%

# THE QUICK COMMERCE JUNKIE
# Quick Commerce > 15%

# THE SHOPAHOLIC
# E-commerce > 15%

# THE INVESTOR
# Investments > 15%

# THE LATE-NIGHT SNACKER
# More than 50% Food Delivery between 21:00 and 02:00

# THE CAB COMMUTER
# Transport > 10%

# THE SUBSCRIPTION LOVER
# 5 or more distinct subscription vendors

# THE YOLO SPENDER
# Savings rate < 10%

# THE DISCIPLINED SAVER
# Savings rate > 40%

##8. Bonus Features
* Run on your own statement



In [ ]:
# Only after the main project works:
# df = pd.read_csv("my_statement.csv")



* Invented archetype



In [ ]:
# Example:
# The Pavement Coffee Connoisseur

cafe_vendors = df[
    df["category"] == "Cafe"
]["vendor_clean"].nunique()

if cafe_vendors >= 4:
    print("THE PAVEMENT COFFEE CONNOISSEUR")



* Day - of - week analysis



In [ ]:
df["day_of_week"] = df["date"].dt.day_name()
day_spending = df[
    df["type"] == "debit"
].groupby("day_of_week")["amount"].sum()

print(day_spending)

day_of_week
Friday       43159.0
Monday       17106.0
Saturday      5260.0
Sunday       62162.0
Thursday     11988.0
Tuesday      35776.0
Wednesday    29859.0
Name: amount, dtype: float64




* Vendor clenup audit



In [ ]:
uncategorised = df[
    df["vendor_clean"] == "Uncategorised"
]

print(
    uncategorised["Description"]
    .unique()
)

['ANI Technologies' 'TWC INDIA' 'BANGALORE ELEC SUPPLY'
 'INSTAMART BANGALORE' 'POS EMPIRE RESTAURANT' 'POS NYKAA BANGALORE'
 'ROPPEN TRANSPORTATION' 'NEFT-TECHCRUSH LABS-SALARY MAY24'
 'RAPIDO BIKE TAXI' 'COFFEE DAY GLOBAL' 'BMS MOVIE TICKETS'
 'INNOVATIVE RETAIL' 'POS THIRD WAVE COFFEE' 'AVENUE SUPERMARTS'
 'POS HP PETROL STATION' 'KIRANAKART TECH' 'POS BANGALORE RESTAURANT'
 'IMPS-RENT-LANDLORD-49966195' 'POS INSTAMART'
 'IMPS-RENT-LANDLORD-35126704' 'POS STARBUCKS BANGALORE']




* Spend forecasting



In [ ]:
df["month"] = df["date"].dt.month
monthly_category = df[
    df["type"] == "debit"
].pivot_table(
    values="amount",
    index="month",
    columns="category",
    aggfunc="sum",
    fill_value=0
)

forecast = monthly_category.tail(3).mean()

print("NEXT MONTH FORECAST")
print(forecast)

NEXT MONTH FORECAST
category
Cash Withdrawal      1000.000000
E-commerce           1364.000000
Entertainment         278.333333
Food Delivery        1616.666667
Groceries             303.333333
Investments          5000.000000
Personal Transfer     876.000000
Quick Commerce        115.666667
Subscriptions           0.000000
Transport             410.666667
Uncategorised        3252.666667
dtype: float64


##9. Tech Stack - What You Can and Cannot Use

In [ ]:
import pandas as pd
import numpy as np

##10. The 7-Day Build Plan
# Day 1
# Parser + Setup

# Day 2
# Vendor Extractor

# Day 3
# Category Tagger + Spending Overview

# Day 4
# Monthly Trends + Time-of-Day

# Day 5
# Anomaly Detection + Archetype Detection

# Day 6
# Final Printed Report

# Day 7
# Polish + Insights + Submit

##11. The Target Printed Report

In [ ]:
# Ensure df has 'hour', 'month', and 'z_score' columns
if 'hour' not in df.columns:
    df["hour"] = df["Time"].astype(str).str[:2].astype(int)
if 'month' not in df.columns:
    df["month"] = df["date"].dt.month
if 'z_score' not in df.columns:
    category_mean = (
        df.groupby("category")["amount"]
        .transform("mean")
    )
    category_std = (
        df.groupby("category")["amount"]
        .transform("std")
    )
    df["z_score"] = (
        (df["amount"] - category_mean)
        / category_std
    )

# Re-define variables from Feature 4 - Spending Overview
credits = df.loc[df["type"] == "credit", "amount"].sum()
debits = df.loc[df["type"] == "debit", "amount"].sum()
net_savings = credits - debits
savings_rate = (
    net_savings / credits * 100
    if credits != 0 else 0
)
top_categories = (
    df[df["type"] == "debit"]
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
top_vendors = (
    df[df["type"] == "debit"]
    .groupby("vendor_clean")["amount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

# Re-define variables for Feature 6 - Time-of-Day Patterns
food_delivery = df[
    df["category"] == "Food Delivery"
]
late_night_food = food_delivery[
    (food_delivery["hour"] >= 21) |
    (food_delivery["hour"] <= 2)
]
if len(food_delivery) > 0:
    late_food_percentage = (
        len(late_night_food)
        / len(food_delivery)
        * 100
    )
else:
    late_food_percentage = 0

# Re-define variables for Feature 5 - Monthly Trend Analysis
month_pivot = df[
    df["type"] == "debit"
].pivot_table(
    values="amount",
    index="month",
    columns="category",
    aggfunc="sum",
    fill_value=0
)

# Re-define variables for Feature 7 - Anomaly Detection
anomalies = df[
    (df["type"] == "debit") &
    (df["z_score"] > 2)
].copy()
anomalies = anomalies.sort_values(
    "z_score",
    ascending=False
)

# Re-define variables for Feature 8 - Spending Archetype Detection
def percentage_for_category(category):
    amount = df.loc[
        (df["type"] == "debit") &
        (df["category"] == category),
        "amount"
    ].sum()
    if debits == 0:
        return 0
    return amount / debits * 100

food_percentage = (
    percentage_for_category("Food Delivery")
    + percentage_for_category("Restaurants")
    + percentage_for_category("Cafe")
)
quick_percentage = percentage_for_category("Quick Commerce")
ecommerce_percentage = percentage_for_category("E-commerce")
investment_percentage = percentage_for_category("Investments")
transport_percentage = percentage_for_category("Transport")

subscription_vendors = df[
    df["category"] == "Subscriptions"
]["vendor_clean"].nunique()

archetypes = []

if food_percentage > 25:
    archetypes.append(
        ("THE FOODIE", food_percentage)
    )
if quick_percentage > 15:
    archetypes.append(
        ("THE QUICK COMMERCE JUNKIE",
         quick_percentage)
    )
if ecommerce_percentage > 15:
    archetypes.append(
        ("THE SHOPAHOLIC",
         ecommerce_percentage)
    )
if investment_percentage > 15:
    archetypes.append(
        ("THE INVESTOR",
         investment_percentage)
    )
if late_food_percentage > 50:
    archetypes.append(
        ("THE LATE-NIGHT SNACKER",
         late_food_percentage)
    )
if transport_percentage > 10:
    archetypes.append(
        ("THE CAB COMMUTER",
         transport_percentage)
    )
if subscription_vendors >= 5:
    archetypes.append(
        ("THE SUBSCRIPTION LOVER",
         subscription_vendors)
    )
if savings_rate < 10:
    archetypes.append(
        ("THE YOLO SPENDER",
         savings_rate)
    )
if savings_rate > 40:
    archetypes.append(
        ("THE DISCIPLINED SAVER",
         savings_rate)
    )

print("=" * 70)
print("SPENDDNA REPORT - RAHUL SHARMA")
print("=" * 70)

print("\nEXECUTIVE SUMMARY")
print("-" * 70)

print(f"Total credits : Rs. {credits:,.2f}")
print(f"Total debits  : Rs. {debits:,.2f}")
print(f"Net change    : Rs. {net_savings:,.2f}")
print(f"Savings rate  : {savings_rate:.2f}%")
print(f"Transactions  : {len(df)}")

print("\nTOP CATEGORIES")
print("-" * 70)

for category, amount in top_categories.items():
    percentage = amount / debits * 100
    print(
        f"{category:<20} "
        f"{percentage:>6.2f}% "
        f"Rs. {amount:>12,.2f}"
    )

print("\nTOP VENDORS")
print("-" * 70)

for vendor, amount in top_vendors.items():
    print(
        f"{vendor:<20} "
        f"Rs. {amount:>12,.2f}"
    )

print("\nTIME-OF-DAY PATTERNS")
print("-" * 70)

print(
    f"Food Delivery after 9 PM: "
    f"{late_food_percentage:.2f}%"
)

print("\nMONTHLY TREND")
print("-" * 70)

print(month_pivot)

print("\nTOP ANOMALIES")
print("-" * 70)

print(
    anomalies[
        [
            "date",
            "vendor_clean",
            "category",
            "amount",
            "z_score"
        ]
    ].head(5).to_string(index=False)
)

print("\nRAHUL'S SPENDING ARCHETYPES")
print("-" * 70)

for name, metric in archetypes:
    print(
        f"-> {name}: {metric:.2f}"
    )

print("=" * 70)

SPENDDNA REPORT - RAHUL SHARMA

EXECUTIVE SUMMARY
----------------------------------------------------------------------
Total credits : Rs. 170,094.00
Total debits  : Rs. 205,310.00
Net change    : Rs. -35,216.00
Savings rate  : -20.70%
Transactions  : 143

TOP CATEGORIES
----------------------------------------------------------------------
Uncategorised         29.43% Rs.    60,430.00
Investments           29.22% Rs.    60,000.00
E-commerce            15.21% Rs.    31,220.00
Food Delivery          9.55% Rs.    19,617.00
Personal Transfer      7.66% Rs.    15,728.00

TOP VENDORS
----------------------------------------------------------------------
Uncategorised        Rs.    60,430.00
Zerodha              Rs.    60,000.00
P2P Transfer         Rs.    15,728.00
Myntra               Rs.    13,629.00
Swiggy               Rs.    12,638.00

TIME-OF-DAY PATTERNS
----------------------------------------------------------------------
Food Delivery after 9 PM: 25.00%

MONTHLY TREND
----------